# Retail sales forecasting: from raw rows to a useful baseline

This notebook demonstrates a compact forecasting workflow: inspect the data, create calendar and lag features, split by time, compare against a seasonal baseline, and explain the model. The approach is deliberately transparent so each modelling choice can be audited.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

DATA = Path('../mock_data/worksheet1_raw_sales_2025.csv')
df = pd.read_csv(DATA, parse_dates=False)
df.head()

## 1. Make the time structure explicit

Calendar features let the model learn recurring patterns without using information from the future. Lagged sales capture momentum; rolling features are shifted first so the current target never leaks into its own predictors.

In [ ]:
df['date'] = pd.Timestamp('2025-01-01') + pd.to_timedelta(df['Day'] - 1, unit='D')
df['day_of_week'] = df['date'].dt.dayofweek
df['week_of_year'] = df['date'].dt.isocalendar().week.astype(int)
df['is_weekend'] = df['day_of_week'].ge(5).astype(int)
df['sales_lag_1'] = df['Sales'].shift(1)
df['sales_lag_7'] = df['Sales'].shift(7)
df['sales_roll_7'] = df['Sales'].shift(1).rolling(7).mean()
df = pd.get_dummies(df, columns=['Weather'], dtype=int)
df = df.dropna().reset_index(drop=True)
df[['date', 'Sales', 'sales_lag_1', 'sales_lag_7', 'sales_roll_7']].head()

## 2. Use a chronological split

Random train/test splits are misleading for forecasting because they allow the model to learn from dates that occur after the supposed prediction point.

In [ ]:
feature_cols = [
    'day_of_week', 'week_of_year', 'is_weekend', 'sales_lag_1',
    'sales_lag_7', 'sales_roll_7', 'Weather_Cloudy', 'Weather_Sunny'
]
split = int(len(df) * 0.8)
train, test = df.iloc[:split], df.iloc[split:]
X_train, y_train = train[feature_cols], train['Sales']
X_test, y_test = test[feature_cols], test['Sales']
print(f'train rows: {len(train)}, test rows: {len(test)}')

## 3. Compare a baseline with a model

The seven-day lag is a strong, interpretable baseline: predict that today will look like the same weekday last week. The gradient-boosting model can combine that signal with weather, calendar, and rolling context.

In [ ]:
baseline = test['sales_lag_7']
model = HistGradientBoostingRegressor(max_iter=150, learning_rate=0.06, max_leaf_nodes=12, random_state=42)
model.fit(X_train, y_train)
pred = model.predict(X_test)

def report(name, actual, predicted):
    return {
        'model': name,
        'MAE': mean_absolute_error(actual, predicted),
        'RMSE': np.sqrt(mean_squared_error(actual, predicted)),
    }

results = pd.DataFrame([
    report('7-day lag baseline', y_test, baseline),
    report('gradient boosting', y_test, pred),
]).sort_values('RMSE')
results

## 4. What this demonstrates

- Forecasting needs a time-aware evaluation design.
- A simple baseline is essential: it tells us whether model complexity adds value.
- Lag and rolling features encode temporal structure without requiring a deep-learning model.
- The next production step would be backtesting across several rolling windows rather than relying on one holdout period.